# Tiny A/B/AB dataset and checker

A collapses an affine chain, B solves one affine equation, and AB needs both skills.

In [4]:
import json
import re
from hashlib import blake2b


def compose(maps, p):
    """Collapse maps listed in application order: g1 first, then g2, ..."""
    A, B = 1, 0
    for a, b in maps:
        A, B = a * A % p, (a * B + b) % p
    return A, B


def apply_chain(maps, x, p):
    for a, b in maps:
        x = (a * x + b) % p
    return x


def invert(a, b, y, p):
    return (y - b) * pow(a, -1, p) % p


def solve(spec):
    if spec["mode"] == "A":
        return compose(spec["maps"], spec["p"])
    A, B = compose(spec["maps"], spec["p"])
    return invert(A, B, spec["y"], spec["p"])


def solve_backwards(spec):
    x = spec["y"]
    for a, b in reversed(spec["maps"]):
        x = invert(a, b, x, spec["p"])
    return x


def render(spec):
    p = spec["p"]
    definitions = "; ".join(
        f"g{i}(z) = ({a}z + {b}) mod {p}"
        for i, (a, b) in enumerate(spec["maps"], 1)
    )
    if spec["mode"] == "A":
        return f"{definitions}. Apply g1 first. Find A, B. FINAL: A=<int> B=<int>"
    chain = "x"
    for i in range(1, len(spec["maps"]) + 1):
        chain = f"g{i}({chain})"
    return f"{definitions}. Given {chain} = {spec['y']}, find x. FINAL: <int>"


FINAL_INT = re.compile(r"FINAL:\s*(-?\d+)")
FINAL_PAIR = re.compile(r"FINAL:\s*A\s*=\s*(-?\d+)\s+B\s*=\s*(-?\d+)")


def parse_final(text, mode):
    matches = (FINAL_PAIR if mode == "A" else FINAL_INT).findall(text)
    if not matches:
        return None
    last = matches[-1]
    return tuple(map(int, last)) if mode == "A" else int(last)


def oracle_check(private_row, model_output):
    spec, gold = private_row["spec"], private_row["gold"]
    answer = parse_final(model_output, spec["mode"])
    if answer is None:
        return False
    if spec["mode"] == "A":
        return tuple(v % spec["p"] for v in answer) == gold
    return answer % spec["p"] == gold


def noisy_accept(oracle_correct, event_id, alpha, beta):
    """Apply IID verifier noise keyed only by a unique rollout event_id."""
    if not 0 <= alpha <= 1 or not 0 <= beta <= 1:
        raise ValueError("alpha and beta must be probabilities")
    u = int.from_bytes(blake2b(event_id.encode(), digest_size=8).digest(), "big") / 2**64
    return {
        "accepted": u < (alpha if oracle_correct else beta),
        "u": u, "oracle_correct": oracle_correct, "alpha": alpha, "beta": beta,
    }


In [5]:
latent_rows = [
    {"problem_id": "A_0", "split": "atomic_A_train", "mode": "A",
     "p": 101, "maps": ((3, 4), (5, 7))},
    {"problem_id": "B_0", "split": "atomic_B_train", "mode": "B",
     "p": 103, "maps": ((7, 11),), "x": 9},
    {"problem_id": "AB_0", "split": "composition_test", "mode": "AB",
     "p": 107, "maps": ((2, 3), (4, 5), (6, 7)), "x": 8},
]

public_rows, private_rows = [], []
for latent in latent_rows:
    spec = dict(latent)
    assert all(a != 0 for a, _ in spec["maps"])
    if spec["mode"] != "A":
        spec["y"] = apply_chain(spec["maps"], spec["x"], spec["p"])
    public_spec = {k: spec[k] for k in ("mode", "p", "maps", "y") if k in spec}
    public_rows.append({
        "problem_id": spec["problem_id"],
        "split": spec["split"],
        "mode": spec["mode"],
        "prompt": render(public_spec),
    })
    private_rows.append({"problem_id": spec["problem_id"], "spec": spec, "gold": solve(spec)})

for row in public_rows:
    print(json.dumps(row))


{"problem_id": "A_0", "split": "atomic_A_train", "mode": "A", "prompt": "g1(z) = (3z + 4) mod 101; g2(z) = (5z + 7) mod 101. Apply g1 first. Find A, B. FINAL: A=<int> B=<int>"}
{"problem_id": "B_0", "split": "atomic_B_train", "mode": "B", "prompt": "g1(z) = (7z + 11) mod 103. Given g1(x) = 74, find x. FINAL: <int>"}
{"problem_id": "AB_0", "split": "composition_test", "mode": "AB", "prompt": "g1(z) = (2z + 3) mod 107; g2(z) = (4z + 5) mod 107; g3(z) = (6z + 7) mod 107. Given g3(g2(g1(x))) = 65, find x. FINAL: <int>"}


In [6]:
private_by_id = {row["problem_id"]: row for row in private_rows}
answers = {"A_0": "FINAL: A=15 B=27", "B_0": "FINAL: 9", "AB_0": "FINAL: 8"}

assert [row["gold"] for row in private_rows] == [(15, 27), 9, 8]
assert solve_backwards(private_by_id["AB_0"]["spec"]) == 8
assert all(oracle_check(private_by_id[row["problem_id"]], answers[row["problem_id"]]) for row in public_rows)
assert oracle_check(private_by_id["AB_0"], "reasoning... FINAL: 115")  # 8 + p is equivalent
assert not oracle_check(private_by_id["AB_0"], "FINAL: 7")
assert not oracle_check(private_by_id["AB_0"], "no final answer")
assert all("gold" not in row and "spec" not in row and "x" not in row for row in public_rows)

train_maps = {tuple(map(tuple, row["spec"]["maps"])) for row in private_rows if row["spec"]["split"] != "composition_test"}
test_maps = {tuple(map(tuple, row["spec"]["maps"])) for row in private_rows if row["spec"]["split"] == "composition_test"}
assert train_maps.isdisjoint(test_maps)

event = "seed7|step0|AB_0|rollout0"
clean_correct = noisy_accept(True, event, alpha=1.0, beta=0.0)
clean_wrong = noisy_accept(False, event, alpha=1.0, beta=0.0)
fn_heavy = noisy_accept(True, event, alpha=0.3, beta=0.0)
assert clean_correct["accepted"] and not clean_wrong["accepted"]
assert fn_heavy == noisy_accept(True, event, alpha=0.3, beta=0.0)  # retry is identical
assert fn_heavy["u"] != noisy_accept(True, event + "-new", 0.3, 0.0)["u"]

print("Exact checker and deterministic noisy verifier passed.")
print(fn_heavy)

Exact checker and deterministic noisy verifier passed.
{'accepted': True, 'u': 0.10244612710685523, 'oracle_correct': True, 'alpha': 0.3, 'beta': 0.0}
